# 02 — Fast Conjunction Search with K-D Trees

The brute-force O(n²) scanner from notebook 01 checks every satellite against every other satellite.
With 14,000 objects that's ~98 million pairs — too slow for real-time use.

This notebook replaces it with a **k-d tree** based approach:

1. What a k-d tree is and why it works for this problem
2. Building a spatial snapshot of all satellite positions
3. Using scipy's KDTree to find candidate pairs efficiently
4. Detailed conjunction analysis on candidates only
5. Benchmarking brute force vs k-d tree
6. Filtering out false positives (docked objects)

This is the algorithm that will run inside the real-time pipeline later.

In [1]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.spatial import KDTree
from sgp4.api import Satrec, jday

print('All imports successful.')

ModuleNotFoundError: No module named 'scipy'

---
## 1. Copy Over Utilities from Notebook 01

In the real pipeline these will live in `src/propagator.py`. For now we redefine them here.

In [ ]:
def propagate(satrec, dt):
    jd, fr = jday(dt.year, dt.month, dt.day,
                  dt.hour, dt.minute, dt.second + dt.microsecond / 1e6)
    err, pos, vel = satrec.sgp4(jd, fr)
    return np.array(pos) if err == 0 else None

def distance_km(pos1, pos2):
    return np.linalg.norm(pos1 - pos2)

# Hardcoded sample TLEs — replace with live fetch once Celestrak is accessible
SAMPLE_TLES = [
    ('ISS (ZARYA)',
     '1 25544U 98067A   24083.54791435  .00018454  00000+0  33038-3 0  9993',
     '2 25544  51.6416 120.8756 0002651 352.5200 113.7897 15.50036278446053'),
    ('CSS (TIANHE)',
     '1 48274U 21035A   24083.54037662  .00022451  00000+0  25480-3 0  9994',
     '2 48274  41.4695 124.4421 0005844 339.6846  20.3951 15.61719547161351'),
    ('STARLINK-1007',
     '1 44713U 19074A   24083.54791435  .00001200  00000+0  10000-3 0  9991',
     '2 44713  53.0000  80.0000 0001200  90.0000 270.0000 15.06000000000001'),
    ('STARLINK-1008',
     '1 44714U 19074B   24083.54791435  .00001210  00000+0  10100-3 0  9992',
     '2 44714  53.0001  80.0010 0001210  90.1000 270.1000 15.06000000000002'),
    ('COSMOS 2251 DEB',
     '1 33791U 93036SX  24083.54791435  .00000500  00000+0  50000-4 0  9995',
     '2 33791  74.0000 200.0000 0050000 180.0000 180.0000 14.80000000000001'),
]

now = datetime.utcnow()
print(f'Reference time: {now.strftime("%Y-%m-%d %H:%M:%S")} UTC')

---
## 2. What is a K-D Tree?

A **k-d tree** (k-dimensional tree) is a data structure that partitions points in k-dimensional space so you can answer nearest-neighbor and radius queries efficiently.

```
Brute force — check every pair:
  n=14,000 satellites → 14000²/2 = 98,000,000 pairs checked
  Time complexity: O(n²)

K-D tree — query radius around each point:
  Build tree once: O(n log n)
  Each query: O(log n)
  For sparse space (most queries return 0 neighbors): ~O(n log n) total
```

The key insight: most satellites are nowhere near each other.
A radius query lets us skip 99.9% of pairs immediately.

Our 3D space is ECI coordinates (x, y, z in km).
We build the tree once per time step and query each satellite against it.

---
## 3. Building a Positional Snapshot

In [ ]:
def build_snapshot(satellite_list, dt):
    """
    Propagate all satellites to time dt.
    Returns:
        names     — list of satellite names
        positions — np.array of shape (n, 3), ECI coords in km
    """
    names = []
    positions = []

    for name, l1, l2 in satellite_list:
        sat = Satrec.twoline2rv(l1, l2)
        pos = propagate(sat, dt)
        if pos is not None:
            names.append(name)
            positions.append(pos)

    return names, np.array(positions)

names, positions = build_snapshot(SAMPLE_TLES, now)

print(f'Snapshot at {now.strftime("%H:%M:%S")} UTC')
print(f'Satellites propagated: {len(names)}')
print()
for name, pos in zip(names, positions):
    alt = np.linalg.norm(pos) - 6371
    print(f'  {name:<20} alt={alt:.0f} km  pos=({pos[0]:.0f}, {pos[1]:.0f}, {pos[2]:.0f})')

---
## 4. K-D Tree Conjunction Search

In [ ]:
def find_candidate_pairs(names, positions, threshold_km=500):
    """
    Use a k-d tree to find all pairs of satellites within threshold_km.
    Returns list of (idx_a, idx_b) index pairs.
    """
    tree = KDTree(positions)

    # For each satellite, find all others within threshold
    # query_ball_tree returns a list of lists: results[i] = [j, k, ...]
    results = tree.query_ball_tree(tree, r=threshold_km)

    pairs = set()
    for i, neighbors in enumerate(results):
        for j in neighbors:
            if j <= i:  # avoid duplicates and self-pairs
                continue
            pairs.add((i, j))

    return list(pairs)

candidates = find_candidate_pairs(names, positions, threshold_km=5000)
print(f'Candidate pairs within 5000 km: {len(candidates)}')
for i, j in candidates:
    d = distance_km(positions[i], positions[j])
    print(f'  {names[i]:<20} — {names[j]:<20}  {d:.1f} km')

---
## 5. Detailed Analysis on Candidates Only

The k-d tree gives us a small set of candidate pairs.
Now we run the full time-series analysis **only on those pairs** to find the actual closest approach.

In [ ]:
def detailed_conjunction_analysis(satellite_list, candidate_pairs_idx,
                                   names, lookahead_minutes=90, step_minutes=1):
    """
    For each candidate pair, compute distance at every time step
    and find the true closest approach.
    """
    # Build Satrec objects indexed by name
    satrecs = {}
    for name, l1, l2 in satellite_list:
        satrecs[name] = Satrec.twoline2rv(l1, l2)

    time_steps = [now + timedelta(minutes=i)
                  for i in range(0, lookahead_minutes, step_minutes)]

    conjunctions = []

    for i, j in candidate_pairs_idx:
        name_a, name_b = names[i], names[j]
        sat_a = satrecs[name_a]
        sat_b = satrecs[name_b]

        min_dist = float('inf')
        min_t = now
        distances_over_time = []

        for t in time_steps:
            pa = propagate(sat_a, t)
            pb = propagate(sat_b, t)
            if pa is None or pb is None:
                continue
            d = distance_km(pa, pb)
            distances_over_time.append(d)
            if d < min_dist:
                min_dist = d
                min_t = t

        conjunctions.append({
            'sat_a': name_a,
            'sat_b': name_b,
            'min_distance_km': round(min_dist, 2),
            'time_of_closest_approach': min_t,
            'minutes_from_now': int((min_t - now).total_seconds() / 60),
            'distances': distances_over_time,
        })

    return sorted(conjunctions, key=lambda x: x['min_distance_km'])

conjunctions = detailed_conjunction_analysis(
    SAMPLE_TLES, candidates, names, lookahead_minutes=90
)

print(f'Conjunction events found: {len(conjunctions)}\n')
for c in conjunctions:
    print(f"  {c['sat_a']:<20} — {c['sat_b']:<20}")
    print(f"    Closest approach : {c['min_distance_km']:.2f} km")
    print(f"    Time             : {c['time_of_closest_approach'].strftime('%H:%M:%S')} UTC")
    print(f"    Minutes from now : {c['minutes_from_now']}")
    print()

---
## 6. Filtering False Positives — Docked Objects

As we saw in notebook 01, docked modules show 0.00 km distance.
We filter these by checking if two objects are already co-located at t=0.

In [ ]:
# Known station complexes — objects physically attached to each other
ISS_COMPLEX = {
    'ISS (ZARYA)', 'POISK', 'ISS (NAUKA)',
    'PROGRESS-MS 32', 'PROGRESS-MS 33',
    'SOYUZ-MS 28', 'CREW DRAGON 12',
}
CSS_COMPLEX = {
    'CSS (TIANHE)', 'CSS (WENTIAN)', 'CSS (MENGTIAN)',
    'TIANZHOU-9', 'SHENZHOU-22',
}
KNOWN_COMPLEXES = [ISS_COMPLEX, CSS_COMPLEX]

def is_same_complex(name_a, name_b):
    for complex_set in KNOWN_COMPLEXES:
        if name_a in complex_set and name_b in complex_set:
            return True
    return False

def is_collocated(positions, i, j, tolerance_km=1.0):
    """Returns True if two objects are already essentially at the same position."""
    return distance_km(positions[i], positions[j]) < tolerance_km

def filter_false_positives(candidate_pairs, names, positions):
    filtered = []
    removed = []
    for i, j in candidate_pairs:
        if is_same_complex(names[i], names[j]):
            removed.append((names[i], names[j], 'known complex'))
        elif is_collocated(positions, i, j):
            removed.append((names[i], names[j], 'collocated at t=0'))
        else:
            filtered.append((i, j))
    return filtered, removed

filtered_pairs, removed = filter_false_positives(candidates, names, positions)

print(f'Pairs before filter : {len(candidates)}')
print(f'Pairs after filter  : {len(filtered_pairs)}')
if removed:
    print(f'\nRemoved ({len(removed)}):')
    for a, b, reason in removed:
        print(f'  {a} — {b}  ({reason})')

---
## 7. Benchmark — Brute Force vs K-D Tree

Let's generate a synthetic catalogue of N satellites and measure the speedup.

In [ ]:
def generate_synthetic_positions(n, altitude_km_range=(300, 2000)):
    """
    Generate n random satellite positions uniformly distributed
    in a spherical shell between two altitudes.
    """
    R_earth = 6371.0
    r_min = R_earth + altitude_km_range[0]
    r_max = R_earth + altitude_km_range[1]

    # Random points on sphere shell
    theta = np.random.uniform(0, 2 * np.pi, n)
    phi = np.arccos(np.random.uniform(-1, 1, n))
    r = np.random.uniform(r_min, r_max, n)

    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)

    return np.column_stack([x, y, z])

def brute_force_search(positions, threshold_km):
    pairs = []
    n = len(positions)
    for i in range(n):
        for j in range(i + 1, n):
            if distance_km(positions[i], positions[j]) < threshold_km:
                pairs.append((i, j))
    return pairs

def kdtree_search(positions, threshold_km):
    tree = KDTree(positions)
    results = tree.query_ball_tree(tree, r=threshold_km)
    pairs = set()
    for i, neighbors in enumerate(results):
        for j in neighbors:
            if j > i:
                pairs.add((i, j))
    return list(pairs)

# Benchmark across increasing catalogue sizes
sizes = [100, 500, 1000, 2000, 5000]
threshold = 500  # km

brute_times = []
kdtree_times = []

for n in sizes:
    pos = generate_synthetic_positions(n)

    if n <= 2000:  # brute force too slow beyond this
        t0 = time.perf_counter()
        brute_force_search(pos, threshold)
        brute_times.append(time.perf_counter() - t0)
    else:
        brute_times.append(None)

    t0 = time.perf_counter()
    kdtree_search(pos, threshold)
    kdtree_times.append(time.perf_counter() - t0)

print(f'{"N":>6}  {"Brute (s)":>12}  {"K-D Tree (s)":>14}  {"Speedup":>10}')
print('-' * 50)
for i, n in enumerate(sizes):
    bt = f'{brute_times[i]:.4f}' if brute_times[i] is not None else '(too slow)'
    kt = f'{kdtree_times[i]:.4f}'
    speedup = f'{brute_times[i]/kdtree_times[i]:.0f}x' if brute_times[i] else '—'
    print(f'{n:>6}  {bt:>12}  {kt:>14}  {speedup:>10}')

In [ ]:
# Plot the benchmark results
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor('#0a0a1a')
fig.patch.set_facecolor('#0a0a1a')

valid_brute = [(s, t) for s, t in zip(sizes, brute_times) if t is not None]
ax.plot([v[0] for v in valid_brute], [v[1] for v in valid_brute],
        color='#ff4444', linewidth=2, marker='o', label='Brute force O(n²)')
ax.plot(sizes, kdtree_times,
        color='#00ff88', linewidth=2, marker='o', label='K-D tree O(n log n)')

ax.set_xlabel('Number of satellites', color='white')
ax.set_ylabel('Time (seconds)', color='white')
ax.set_title('Brute Force vs K-D Tree Conjunction Search', color='white', fontsize=13)
ax.tick_params(colors='white')
ax.grid(True, alpha=0.2, color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.show()

---
## 8. The Full Fast Pipeline

Putting it all together — this is the function that will run inside the real-time detector service.

In [ ]:
def fast_conjunction_scan(satellite_list, lookahead_minutes=90,
                          step_minutes=1, screen_threshold_km=500,
                          report_threshold_km=50):
    """
    Full fast conjunction pipeline:
      1. Build positional snapshot at t=now
      2. K-D tree screen for candidate pairs
      3. Filter false positives (docked objects)
      4. Detailed time-series analysis on candidates
      5. Return only events below report_threshold_km

    screen_threshold_km : coarse filter — anything beyond this is ignored
    report_threshold_km : fine filter — only these events are returned
    """
    t_start = time.perf_counter()

    # Step 1: snapshot
    names, positions = build_snapshot(satellite_list, now)
    print(f'[1] Snapshot built     : {len(names)} satellites')

    # Step 2: k-d tree screen
    candidates = find_candidate_pairs(names, positions, screen_threshold_km)
    print(f'[2] Candidate pairs    : {len(candidates)}')

    # Step 3: filter
    candidates, removed = filter_false_positives(candidates, names, positions)
    print(f'[3] After filter       : {len(candidates)} ({len(removed)} removed)')

    # Step 4: detailed analysis
    conjunctions = detailed_conjunction_analysis(
        satellite_list, candidates, names, lookahead_minutes, step_minutes
    )
    print(f'[4] Conjunctions found : {len(conjunctions)}')

    # Step 5: apply report threshold
    reported = [c for c in conjunctions if c['min_distance_km'] < report_threshold_km]
    print(f'[5] Events to report   : {len(reported)} (below {report_threshold_km} km)')
    print(f'    Total time         : {time.perf_counter() - t_start:.3f}s')

    return reported

print('Running fast conjunction scan...\n')
events = fast_conjunction_scan(
    SAMPLE_TLES,
    lookahead_minutes=90,
    screen_threshold_km=5000,
    report_threshold_km=5000,  # wide for sample data — tighten with real catalogue
)

if events:
    print()
    df = pd.DataFrame([{k: v for k, v in e.items() if k != 'distances'} for e in events])
    print(df.to_string(index=False))
else:
    print('\nNo reportable events in this window.')

---
## 9. What We Have & What's Next

### What this notebook established:
- K-D tree based spatial indexing cuts search from O(n²) to O(n log n)
- Two-stage pipeline: coarse screen → detailed analysis on candidates only
- False positive filtering for docked/collocated objects
- `fast_conjunction_scan()` is the function the real pipeline will call

### What's next — Notebook 03:
Right now our output is just a distance in km. The next step is to train an **ML model** that takes a conjunction event and outputs a **collision probability score** (0.0–1.0), accounting for:
- Miss distance
- Closing velocity
- Object sizes (radar cross section as a proxy)
- TLE age (older = less reliable = higher uncertainty)
- Orbital altitude (higher atmosphere = less drag uncertainty)

### Full build order:
```
✅ 01_explore_tle_data.ipynb         
✅ 02_fast_conjunction_search.ipynb
⬜ 03_ml_model_training.ipynb        (risk score model)
⬜ src/ingester.py                   (Kafka producer)
⬜ src/propagator.py                 (Kafka consumer -> Redis)
⬜ src/detector.py                   (conjunction service)
⬜ dashboard/                        (CesiumJS frontend)
```